In [1]:
import sys
from pathlib import Path

sys.path.append(f"{Path().absolute().parent}")

In [2]:
import warnings

# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning, module="gpytorch")

In [3]:
from apps.mobility_robustness_optimization.mobility_robustness_optimization import *
from apps.mobility_robustness_optimization.simple_mro import SimpleMRO

In [4]:
params = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 100,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 5,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 5,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 5,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.5,
                    "variance": 0.8,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

In [5]:
topology = pd.read_csv("data/sim_data/topology.csv")
ue_data = pd.read_csv('data/sim_data/UE_Data_20UE_100ticks.csv')

topology.loc[topology["cell_id"] == "cell_1", "cell_lat"] = -90
topology.loc[topology["cell_id"] == "cell_2", "cell_lat"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lat"] = 90

topology.loc[topology["cell_id"] == "cell_1", "cell_lon"] = -180
topology.loc[topology["cell_id"] == "cell_2", "cell_lon"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lon"] = 180

topology.loc[topology["cell_id"] == "cell_1", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_2", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_3", "cell_carrier_freq_mhz"] = 2100

topology

,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz
0,-90.0,-180.0,cell_1,0,2100
1,0.0,0.0,cell_2,120,2100
2,90.0,180.0,cell_3,240,2100


In [6]:
mro = SimpleMRO(params, topology)

In [7]:
mro.update(ue_data)

No Bayesian Digital Twins available for update. Training from scratch.


[2025-04-30 12:51:40,098] INFO:  Iter 1/100 - Loss: 0.784 (delta=inf)
[2025-04-30 12:51:40,415] INFO:  Iter 2/100 - Loss: 0.765 (delta=-0.019082)
[2025-04-30 12:51:40,740] INFO:  Iter 3/100 - Loss: 0.745 (delta=-0.019103)
[2025-04-30 12:51:41,029] INFO:  Iter 4/100 - Loss: 0.726 (delta=-0.019145)
[2025-04-30 12:51:41,411] INFO:  Iter 5/100 - Loss: 0.707 (delta=-0.019228)
[2025-04-30 12:51:41,868] INFO:  Iter 6/100 - Loss: 0.688 (delta=-0.019358)
[2025-04-30 12:51:42,295] INFO:  Iter 7/100 - Loss: 0.668 (delta=-0.019516)
[2025-04-30 12:51:42,613] INFO:  Iter 8/100 - Loss: 0.648 (delta=-0.019730)
[2025-04-30 12:51:42,895] INFO:  Iter 9/100 - Loss: 0.629 (delta=-0.019933)
[2025-04-30 12:51:43,294] INFO:  Iter 10/100 - Loss: 0.608 (delta=-0.020149)
[2025-04-30 12:51:43,574] INFO:  Iter 11/100 - Loss: 0.588 (delta=-0.020328)
[2025-04-30 12:51:43,854] INFO:  Iter 12/100 - Loss: 0.567 (delta=-0.020553)
[2025-04-30 12:51:44,196] INFO:  Iter 13/100 - Loss: 0.547 (delta=-0.020729)
[2025-04-30 12

In [8]:
hyst,ttt = mro.solve()

Epoch  Hyst           TTT    MRO Metric  
-----------------------------------------
0      2.7084872630   26     99.750000   
1      4.1826148152   77     99.400000   
2      3.9033776263   56     99.450000   
3      0.0464854026   44     99.650000   
4      3.2066818773   35     99.650000   
5      0.8825025307   61     99.450000   
6      2.2475719598   75     99.450000   
7      1.3935350212   39     99.650000   
8      2.8337031227   73     99.450000   
9      1.2529481522   89     99.250000   
10     1.9481560803   14     99.950000   
11     2.8565296148   95     99.250000   
12     0.6018822805   12     99.950000   
13     1.3960662511   100    99.250000   
14     1.1550067680   72     99.450000   
15     3.5370754617   28     99.750000   
16     1.1774771643   85     99.300000   
17     2.0229587480   52     99.500000   
18     1.8056813817   34     99.650000   
19     2.3380976549   16     99.850000   
20     4.0860156822   39     99.650000   
21     1.1149276462   88     99.25